In [0]:
"""
Prepare DQ payload for OpenMetadata publishing.

This first notebook reads the latest data-quality summary rows from Unity Catalog
summary tables and passes them to a second notebook as a JSON payload.
It avoids writing an intermediate summary table and keeps the access path
focused on the original Unity Catalog tables.
"""
import json
import os
import time

# ---------------------------------------------------------------------------
# Parameters
# ---------------------------------------------------------------------------
dbutils.widgets.text("norm_project", "")
dbutils.widgets.text("norm_job", "")
dbutils.widgets.text("omd_fqn", "")
dbutils.widgets.text("omd_base_url", "https://openmetadata.intranet.geodis.org/api")
dbutils.widgets.text("omd_push_notebook_path", "")
dbutils.widgets.text("payload_json", "")

NORM_PROJECT = dbutils.widgets.get("norm_project")
NORM_JOB = dbutils.widgets.get("norm_job")
_raw_fqn = dbutils.widgets.get("omd_fqn").strip()
OMD_FQN = (" " + _raw_fqn) if _raw_fqn.startswith("starburst-uat01") else _raw_fqn
# Token read from environment variable — never passed as a task parameter
OMD_TOKEN = os.getenv("OMD_TOKEN", "")
OMD_BASE_URL = dbutils.widgets.get("omd_base_url").rstrip("/")
OMD_PUSH_NOTEBOOK_PATH = (dbutils.widgets.get("omd_push_notebook_path") or os.getenv("OMD_NOTEBOOK_PATH", "")).strip()

CATALOG = "dbr_uat01_gdm"
RESULTS_SCHEMA = "db_gdm_dq_results"
SUMMARY_COLUMNS_TABLE = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{NORM_PROJECT}_{NORM_JOB}_summary_columns"
SUMMARY_DATASET_TABLE = f"{CATALOG}.{RESULTS_SCHEMA}.dq_{NORM_PROJECT}_{NORM_JOB}_summary_dataset"

print("=== Prepare DQ payload for OpenMetadata ===")
print(f"Project : {NORM_PROJECT}")
print(f"Job     : {NORM_JOB}")
print(f"FQN     : {OMD_FQN}")
print(f"Push notebook: {OMD_PUSH_NOTEBOOK_PATH or '<not provided>'}")

if not OMD_FQN:
    raise ValueError("omd_fqn parameter is empty")


def _fetch_summary_rows():
    """Read the latest DQ summary rows directly from Unity Catalog summary tables."""
    col_query = f"""
        SELECT column_name, check_type, score, status, faulty_count, total_rows
        FROM {SUMMARY_COLUMNS_TABLE}
        WHERE CAST(execution_timestamp AS TIMESTAMP) = (
            SELECT MAX(CAST(execution_timestamp AS TIMESTAMP))
            FROM {SUMMARY_COLUMNS_TABLE}
        )
    """
    ds_query = f"""
        SELECT check_type, dataset_score, dataset_status
        FROM {SUMMARY_DATASET_TABLE}
        WHERE CAST(execution_timestamp AS TIMESTAMP) = (
            SELECT MAX(CAST(execution_timestamp AS TIMESTAMP))
            FROM {SUMMARY_DATASET_TABLE}
        )
    """

    columns_rows = [row.asDict() for row in spark.sql(col_query).collect()]
    dataset_rows = [row.asDict() for row in spark.sql(ds_query).collect()]
    return columns_rows, dataset_rows


print("\n[1/3] Reading latest DQ results from Unity Catalog summary tables...")
columns_rows, dataset_rows = _fetch_summary_rows()
print(f"  → {len(columns_rows)} column rows, {len(dataset_rows)} dataset rows")

if not columns_rows:
    raise ValueError("No DQ results found in summary tables — run the DQ job first")

payload = {
    "source": "unity_catalog_summary",
    "norm_project": NORM_PROJECT,
    "norm_job": NORM_JOB,
    "omd_fqn": OMD_FQN,
    "omd_base_url": OMD_BASE_URL,
    "columns_rows": columns_rows,
    "dataset_rows": dataset_rows,
    "generated_at": int(time.time() * 1000),
}

payload_json = json.dumps(payload)
print("\n[2/3] Payload prepared successfully")
print(f"  → payload size: {len(payload_json)} chars")

payload_dir = "/dbfs/tmp/omd_payload"
dbutils.fs.mkdirs(payload_dir)
payload_path = f"{payload_dir}/{NORM_PROJECT}_{NORM_JOB}.json"
dbutils.fs.put(payload_path, payload_json, overwrite=True)
print(f"  → Wrote payload to {payload_path}")

# Publish the payload to downstream tasks using Databricks task values and a DBFS path.
try:
    dbutils.jobs.taskValues.set("payload_json", payload_json)
    dbutils.jobs.taskValues.set("payload_path", payload_path)
    dbutils.jobs.taskValues.set("push_notebook", OMD_PUSH_NOTEBOOK_PATH)
    print("  → Published payload to downstream task values")
except Exception as exc:
    print(f"  → Task values unavailable in this context: {exc}")

# The second task will receive this payload through the DBFS path and task values.
dbutils.notebook.exit(json.dumps({
    "status": "success",
    "payload_rows": len(columns_rows),
    "payload_json": payload_json,
    "payload_path": payload_path,
    "push_notebook": OMD_PUSH_NOTEBOOK_PATH,
}))
